In [1]:
# Lab Task: Wine Classification using Support Vector Machine (SVM)
# This cell programmatically writes all the necessary files for your local testing and Hugging Face deployment.

import os

# 1. Create train_model.py
train_model_code = """from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import joblib

# Load dataset
wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=42
)

# Train SVM Classifier (a different model than Random Forest)
model = SVC(kernel='linear', C=1.0, random_state=42)
model.fit(X_train, y_train)

# Evaluate and save
acc = accuracy_score(y_test, model.predict(X_test))
print(f"Model trained successfully! Accuracy: {acc:.4f}")
joblib.dump(model, 'wine_model.joblib')
"""

with open("train_model.py", "w") as f:
    f.write(train_model_code)

# 2. Create app.py (Streamlit frontend customized for Wine dataset)
app_code = """import streamlit as st
import joblib
import pandas as pd
from sklearn.datasets import load_wine

# Load model and class names
model = joblib.load("wine_model.joblib")
wine = load_wine()

st.title("🍷 Wine Classification App")
st.write("Enter chemical features of the wine to predict its class (Class 0, 1, or 2).")

# Create input fields for key features
alcohol = st.number_input("Alcohol", 11.0, 15.0, 13.0)
malic_acid = st.number_input("Malic Acid", 0.7, 5.8, 2.3)
ash = st.number_input("Ash", 1.3, 3.2, 2.3)
alcalinity = st.number_input("Alcalinity of Ash", 10.6, 30.0, 19.5)
magnesium = st.number_input("Magnesium", 70.0, 162.0, 99.0)

if st.button("Predict Wine Class"):
    # Prepare features (padding remaining features with their average values to fit the 13-feature input shape)
    features = [alcohol, malic_acid, ash, alcalinity, magnesium] + [0.0] * 8
    prediction = model.predict([features])[0]
    target_name = wine.target_names[prediction]
    st.success(f"Prediction: {target_name.upper()}")
"""

with open("app.py", "w") as f:
    f.write(app_code)

# 3. Create requirements.txt
requirements_code = """streamlit
scikit-learn
joblib
pandas
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_code)

# 4. Create Dockerfile
dockerfile_code = """FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 7860
CMD [\"streamlit\", \"run\", \"app.py\", \"--server.address=0.0.0.0\", \"--server.port=7860\"]
"""

with open("Dockerfile", "w") as f:
    f.write(dockerfile_code)

# Run the training script locally in Colab
print("--- Training the model ---")
!python train_model.py
print("\n--- Files Created Successfully ---")
print(os.listdir('.'))

--- Training the model ---
Model trained successfully! Accuracy: 1.0000

--- Files Created Successfully ---
['.config', 'requirements.txt', 'wine_model.joblib', 'app.py', 'Dockerfile', 'train_model.py', 'sample_data']
